Do the interpolation for all 20 days. Use basic pandas linear interpolation and save a summary of where there was 0's located

In [1]:
# Left eye raw data

import pandas as pd
import numpy as np

input_file = "L:/Auditdata/CONNECT-ME/Nikolai/pupillometry/Data/Left_light_off_250.xlsx"

output_clean_file = "L:/Auditdata/CONNECT-ME/DTU/FrederikWeinan_Thesis/Interpolate_days/Left_light_off_250_interpolated.xlsx"
output_summary_file = "L:/Auditdata/CONNECT-ME\DTU/FrederikWeinan_Thesis/Interpolate_days/Interpolation_summary.xlsx"


xls = pd.ExcelFile(input_file)
sheet_names = xls.sheet_names

writer_clean = pd.ExcelWriter(output_clean_file, engine="openpyxl")
summary_rows = []


for sheet in sheet_names:
    print(f"Processing {sheet}")

    df = pd.read_excel(xls, sheet_name=sheet)

    # We find the Time row because everything under is time series
    time_row_idx = df.index[df.iloc[:, 0].astype(str).str.strip() == "Time"][0]

    # Split based on where Time row is
    meta = df.iloc[:time_row_idx + 1].copy()
    signal = df.iloc[time_row_idx + 1:].copy()

    pupil_cols = signal.columns[1:]

    for col in pupil_cols:

        # force numeric
        signal[col] = pd.to_numeric(signal[col], errors="coerce")

        col_series = signal[col]

        is_glass_eye = col_series.fillna(0).eq(0).all()
        zero_count = (col_series == 0).sum()

        # Check if everything is 0, because then it is glass eye (Some places also missing all data, but this will be classed as the same)
        if is_glass_eye:
            summary_rows.append({
                "Sheet": sheet,
                "Participant": col,
                "Zeros_Replaced": 0,
                "Glass_Eye": True
            })
            continue

        # interpolate pipeline
        signal[col] = col_series.replace(0, np.nan)
        signal[col] = signal[col].interpolate(method="linear")
        signal[col] = signal[col].bfill()
        signal[col] = signal[col].ffill()

        summary_rows.append({
            "Sheet": sheet,
            "Participant": col,
            "Zeros_Replaced": int(zero_count),
            "Glass_Eye": False
        })

    cleaned = pd.concat([meta, signal], ignore_index=False)
    cleaned.to_excel(writer_clean, sheet_name=sheet, index=False)

writer_clean.close()


summary_df = pd.DataFrame(summary_rows)
with pd.ExcelWriter(output_summary_file, engine="openpyxl") as writer_summary:
    summary_df.to_excel(writer_summary, sheet_name="summary", index=False)

print("DONE")

<>:9: SyntaxWarning: invalid escape sequence '\D'
<>:9: SyntaxWarning: invalid escape sequence '\D'
C:\Users\FLUN0029\AppData\Local\Temp\ipykernel_14192\1927158153.py:9: SyntaxWarning: invalid escape sequence '\D'
  output_summary_file = "L:/Auditdata/CONNECT-ME\DTU/FrederikWeinan_Thesis/Interpolate_days/Interpolation_summary.xlsx"


Processing Day_1_L
Processing Follow up_1
Processing Follow up_2
Processing Follow up_3
Processing Follow up_4
Processing Follow up_5
Processing Follow up_6
Processing Follow up_7
Processing Follow up_8
Processing Follow up_9
Processing Follow up_10
Processing Follow up_11
Processing Follow up_12
Processing Follow up_13
Processing Follow up_14
Processing Follow up_15
Processing Follow up_16
Processing Follow up_17
Processing Follow up_18
Processing Follow up_19
DONE


In [2]:
# Right eye raw data (Full copy of above, with new paths)

import pandas as pd
import numpy as np


input_file = "L:/Auditdata/CONNECT-ME/Nikolai/pupillometry/Data/Right_light_off_250.xlsx"

output_clean_file = "L:/Auditdata/CONNECT-ME/DTU/FrederikWeinan_Thesis/Interpolate_days/Right_light_off_250_interpolated.xlsx"
output_summary_file = "L:/Auditdata/CONNECT-ME\DTU/FrederikWeinan_Thesis/Interpolate_days/Interpolation_summary_Right.xlsx"


xls = pd.ExcelFile(input_file)
sheet_names = xls.sheet_names

writer_clean = pd.ExcelWriter(output_clean_file, engine="openpyxl")

summary_rows = []

for sheet in sheet_names:
    print(f"Processing {sheet}")

    df = pd.read_excel(xls, sheet_name=sheet)
    time_row_idx = df.index[df.iloc[:, 0].astype(str).str.strip() == "Time"][0]
    meta = df.iloc[:time_row_idx + 1].copy()
    signal = df.iloc[time_row_idx + 1:].copy()

    pupil_cols = signal.columns[1:]

    for col in pupil_cols:
        signal[col] = pd.to_numeric(signal[col], errors="coerce")

        col_series = signal[col]

        is_glass_eye = col_series.fillna(0).eq(0).all()
        zero_count = (col_series == 0).sum()

        if is_glass_eye:
            summary_rows.append({
                "Sheet": sheet,
                "Participant": col,
                "Zeros_Replaced": 0,
                "Glass_Eye": True
            })
            continue
        signal[col] = col_series.replace(0, np.nan)
        signal[col] = signal[col].interpolate(method="linear")
        signal[col] = signal[col].bfill()
        signal[col] = signal[col].ffill()

        summary_rows.append({
            "Sheet": sheet,
            "Participant": col,
            "Zeros_Replaced": int(zero_count),
            "Glass_Eye": False
        })
    cleaned = pd.concat([meta, signal], ignore_index=False)
    cleaned.to_excel(writer_clean, sheet_name=sheet, index=False)

writer_clean.close()

summary_df = pd.DataFrame(summary_rows)
with pd.ExcelWriter(output_summary_file, engine="openpyxl") as writer_summary:
    summary_df.to_excel(writer_summary, sheet_name="summary", index=False)

print("DONE")

<>:10: SyntaxWarning: invalid escape sequence '\D'
<>:10: SyntaxWarning: invalid escape sequence '\D'
C:\Users\FLUN0029\AppData\Local\Temp\ipykernel_14192\1322507154.py:10: SyntaxWarning: invalid escape sequence '\D'
  output_summary_file = "L:/Auditdata/CONNECT-ME\DTU/FrederikWeinan_Thesis/Interpolate_days/Interpolation_summary_Right.xlsx"


Processing Day_1_R
Processing Follow up_1
Processing Follow up_2
Processing Follow up_3
Processing Follow up_4
Processing Follow up_5
Processing Follow up_6
Processing Follow up_7
Processing Follow up_8
Processing Follow up_9
Processing Follow up_10
Processing Follow up_11
Processing Follow up_12
Processing Follow up_13
Processing Follow up_14
Processing Follow up_15
Processing Follow up_16
Processing Follow up_17
Processing Follow up_18
Processing Follow up_19
DONE


# Check for CH errors

In [3]:
import pandas as pd
import numpy as np

left_df = pd.read_csv("../Modified_Pupilometri/pupilometry_features_left.csv")
right_df = pd.read_csv("../Modified_Pupilometri/pupilometry_features_right.csv")

# Add eye label explicitly
left_df["eye_dataset"] = "left"
right_df["eye_dataset"] = "right"

# Combine
all_df = pd.concat([left_df, right_df], ignore_index=True)


# Ignore rows where measurements are missing
required_cols = ["pupil_size", "pupil_min", "ch"]

clean_df = all_df.dropna(subset=required_cols).copy()

# Convert to numeric safely
for col in required_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

clean_df = clean_df.dropna(subset=required_cols)


clean_df["calculated_ch"] = (
    ((clean_df["pupil_size"] - clean_df["pupil_min"])
     / clean_df["pupil_size"]) * 100
).round().abs()


clean_df["ch_error"] = (
    clean_df["ch"] - clean_df["calculated_ch"]
).round(3)


# Small tolerance for floating-point safety
mismatch_df = clean_df[np.abs(clean_df["ch_error"]) > 0.05].copy()

result_df = mismatch_df[[
    "record_id",
    "redcap_repeat_instance",
    "eye_dataset",
    "pupil_size",
    "pupil_min",
    "ch",
    "calculated_ch",
    "ch_error"
]].sort_values([
    "record_id",
    "redcap_repeat_instance"
])

result_df["redcap_repeat_instance"] = (
    result_df["redcap_repeat_instance"] - 1
)

print("====================================")
print(f"Total checked rows: {len(clean_df)}")
print(f"Rows with CH mismatches: {len(result_df)}")
print("====================================")

# print("\nALL CH MISMATCHES:\n")
# print(result_df.to_string(index=False)) # Privacy

Total checked rows: 1944
Rows with CH mismatches: 20
